In [12]:
!awk '/>/{$0=$1}1' 02_combined_aa/cached_aa.fasta > 03_headers_aa/nitrogen.fasta

In [10]:
!awk '/>/{$0=$1}1' 02_combined_nt/cached_nt.fasta > 03_headers_nt/nitrogen.fasta

In [3]:
!mkdir 04_alignment


In [13]:
!mafft --auto 03_headers_aa/nitrogen.fasta > 04_alignment/nitrogen.fasta

nthread = 0
nthreadpair = 0
nthreadtb = 0
ppenalty_ex = 0
stacksize: 8192 kb
rescale = 1
Gap Penalty = -1.53, +0.00, +0.00



Making a distance matrix ..

There are 9 ambiguous characters.
 1801 / 1863
done.

Constructing a UPGMA tree (efffree=0) ... 
 1860 / 1863
done.

Progressive alignment 1/2... 
STEP  1801 / 1862 
Reallocating..done. *alloclen = 1301

done.

Making a distance matrix from msa.. 
 1800 / 1863
done.

Constructing a UPGMA tree (efffree=1) ... 
 1860 / 1863
done.

Progressive alignment 2/2... 
STEP  1801 / 1862 
Reallocating..done. *alloclen = 1302

done.

disttbfast (aa) Version 7.520
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
0 thread(s)


Strategy:
 FFT-NS-2 (Fast but rough)
 Progressive method (guide trees were built 2 times.)

If unsure which option to use, try 'mafft --auto input > output'.
For more information, see 'mafft --help', 'mafft --man' and the mafft page.

The default gap scoring scheme has been changed in version 7.110 (2013 Oct).
It 

In [8]:
!mkdir 05_tcoffee 

In [17]:
from Bio import SeqIO

In [18]:
input_foil = "03_headers_nt/nitrogen.fasta"
output_foil = "03_headers_nt/removed.fasta"


with open(output_foil, "w") as out_handle:
    for record in SeqIO.parse(input_file, "fasta"):
        original_length = len(record.seq)
        remainder = original_length % 3

        if remainder != 0:
            print(f"Trimming {remainder} base(s) from {record.id} (original length: {original_length})")
            record.seq = record.seq[:-remainder]
        else:
            print(f"No trimming needed for {record.id} (length: {original_length})")

        SeqIO.write(record, out_handle, "fasta")

No trimming needed for tr|Q1K0H8|Q1K0H8_DESA6 (length: 339)
No trimming needed for tr|Q1JYN3|Q1JYN3_DESA6 (length: 339)
No trimming needed for tr|Q1K3N7|Q1K3N7_DESA6 (length: 339)
No trimming needed for tr|I4DAS9|I4DAS9_DESAJ (length: 318)
No trimming needed for tr|I4D1H6|I4D1H6_DESAJ (length: 339)
No trimming needed for tr|I4DAS8|I4DAS8_DESAJ (length: 384)
No trimming needed for tr|I4DBX9|I4DBX9_DESAJ (length: 339)
No trimming needed for tr|G7WJI3|G7WJI3_DESOD (length: 318)
No trimming needed for tr|G7WJN1|G7WJN1_DESOD (length: 339)
No trimming needed for tr|G7WA29|G7WA29_DESOD (length: 339)
No trimming needed for tr|G7WJI2|G7WJI2_DESOD (length: 384)
No trimming needed for tr|G7WCB5|G7WCB5_DESOD (length: 339)
No trimming needed for tr|W0E589|W0E589_MARPU (length: 339)
No trimming needed for tr|W0DV63|W0DV63_MARPU (length: 339)
No trimming needed for tr|A0A1I0CAP3|A0A1I0CAP3_9FIRM (length: 348)
No trimming needed for tr|J7IWQ9|J7IWQ9_DESMD (length: 339)
No trimming needed for tr|J7J4J5

In [20]:
input_file = "03_headers_nt/removed.fasta"
output = "03_headers_nt/removed2.fasta"

stop_codons = {"TAA", "TAG", "TGA"}

with open(output, "w") as out_handle:
    for record in SeqIO.parse(input_file, "fasta"):
        seq = str(record.seq).upper()
        if len(seq) >= 3 and seq[-3:] in stop_codons:
            print(f"Removing stop codon from {record.id}: {seq[-3:]}")
            record.seq = record.seq[:-3]
        else:
            print(f"No stop codon found at end of {record.id}")
        SeqIO.write(record, out_handle, "fasta")

Removing stop codon from tr|Q1K0H8|Q1K0H8_DESA6: TAA
Removing stop codon from tr|Q1JYN3|Q1JYN3_DESA6: TAA
Removing stop codon from tr|Q1K3N7|Q1K3N7_DESA6: TAA
Removing stop codon from tr|I4DAS9|I4DAS9_DESAJ: TAA
Removing stop codon from tr|I4D1H6|I4D1H6_DESAJ: TAA
Removing stop codon from tr|I4DAS8|I4DAS8_DESAJ: TGA
Removing stop codon from tr|I4DBX9|I4DBX9_DESAJ: TAA
Removing stop codon from tr|G7WJI3|G7WJI3_DESOD: TAA
Removing stop codon from tr|G7WJN1|G7WJN1_DESOD: TAA
Removing stop codon from tr|G7WA29|G7WA29_DESOD: TAA
Removing stop codon from tr|G7WJI2|G7WJI2_DESOD: TAG
Removing stop codon from tr|G7WCB5|G7WCB5_DESOD: TAG
Removing stop codon from tr|W0E589|W0E589_MARPU: TAG
Removing stop codon from tr|W0DV63|W0DV63_MARPU: TGA
Removing stop codon from tr|A0A1I0CAP3|A0A1I0CAP3_9FIRM: TAA
Removing stop codon from tr|J7IWQ9|J7IWQ9_DESMD: TAA
Removing stop codon from tr|J7J4J5|J7J4J5_DESMD: TAA
Removing stop codon from tr|J7IQD3|J7IQD3_DESMD: TAA
Removing stop codon from tr|J7IVU6|J7I

In [12]:
!t_coffee -other_pg seq_reformat -in 03_headers_nt/removed2.fasta -in2 04_alignment/nitrogen.fasta -action +thread_dna_on_prot_aln -output fasta > 05_tcoffee/nitrogen.fasta


